In [5]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cftime
import os

# ==========================================
# 1. SETUP: SELECT THE CORRECT SEASON
# ==========================================
# If you want August dates, you MUST load the SUMMER file.
# If you want November dates, load the FALL file.
file_path = '/work/Soelem.Bhuiyan/spi/spi_outputs_ensemble_mean_hist/SPI3_USA_EnsembleMean_Spring_1921-2010.nc' 
output_folder = "frames_spi_usa_spring_ensemble"

os.makedirs(output_folder, exist_ok=True)

print(f"Loading {file_path}...")
ds = xr.open_dataset(file_path)
spi_da = ds['spi3']

# ==========================================
# 2. DEFINE TIMESTEPS
# ==========================================
# NOTE: Ensure these years actually exist in your file!
# If your file is named "1921-2010", you cannot plot 2011-2019.
# I have adjusted this list to years that likely exist in your 1921-2010 file.

date_list = [
    cftime.DatetimeJulian(2001, 5, 15), # Using day=15 is safer for monthly data
    cftime.DatetimeJulian(2002, 5, 15),
    cftime.DatetimeJulian(2003, 5, 15),
    cftime.DatetimeJulian(2004, 5, 15),
    cftime.DatetimeJulian(2005, 5, 15),
    cftime.DatetimeJulian(2006, 5, 15),
    cftime.DatetimeJulian(2007, 5, 15),
    cftime.DatetimeJulian(2008, 5, 15),
    cftime.DatetimeJulian(2009, 5, 15),
    cftime.DatetimeJulian(2010, 5, 15),
]

print(f"Generating {len(date_list)} frames...")

# ==========================================
# 3. GENERATE FRAMES
# ==========================================
for i, target_date in enumerate(date_list):
    try:
        # A. Select Data
        # We REMOVED tolerance="10D". 'nearest' is sufficient for yearly data.
        try:
            data_slice = spi_da.sel(time=target_date, method="nearest")
        except KeyError:
            print(f"  SKIPPING {target_date}: Date not found in file range!")
            continue

        # B. Setup Map
        fig = plt.figure(figsize=(12, 6), dpi=150)
        ax = plt.axes(projection=ccrs.PlateCarree())
        
        # Add Coastlines & States
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5)
        ax.add_feature(cfeature.STATES, linewidth=0.3)
        
        # C. Plot Data
        plot = data_slice.plot(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap='BrBG',
            vmin=-2.5, vmax=2.5,
            add_labels=False,
            cbar_kwargs={'label': 'SPI-3 Index', 'shrink': 0.8}
        )
        
        # D. Add Title
        month_name = target_date.strftime("%B") 
        # Use data_slice.time.dt.year to get the ACTUAL year from the file, 
        # just in case 'nearest' grabbed a neighbor (though unlikely here)
        actual_year = int(data_slice.time.dt.year)
        ax.set_title(f"USA {month_name} Drought Index (SPI-3)\n{actual_year}", fontsize=14, fontweight='bold')
        
        # E. Save Frame
        filename = os.path.join(output_folder, f"frame_{i:03d}.png")
        plt.savefig(filename, bbox_inches='tight')
        plt.close(fig)
        
        print(f"Saved {filename} ({actual_year}-{target_date.month:02d})")
        
    except Exception as e:
        print(f"Error processing {target_date}: {e}")

print("Done.")

Loading /work/Soelem.Bhuiyan/spi/spi_outputs_ensemble_mean_hist/SPI3_USA_EnsembleMean_Spring_1921-2010.nc...
Generating 10 frames...
Saved frames_spi_usa_spring_ensemble/frame_000.png (2001-05)
Saved frames_spi_usa_spring_ensemble/frame_001.png (2002-05)
Saved frames_spi_usa_spring_ensemble/frame_002.png (2003-05)
Saved frames_spi_usa_spring_ensemble/frame_003.png (2004-05)
Saved frames_spi_usa_spring_ensemble/frame_004.png (2005-05)
Saved frames_spi_usa_spring_ensemble/frame_005.png (2006-05)
Saved frames_spi_usa_spring_ensemble/frame_006.png (2007-05)
Saved frames_spi_usa_spring_ensemble/frame_007.png (2008-05)
Saved frames_spi_usa_spring_ensemble/frame_008.png (2009-05)
Saved frames_spi_usa_spring_ensemble/frame_009.png (2010-05)
Done.


In [6]:
from PIL import Image
import os
import glob

# ==========================================
# 1. SETUP
# ==========================================
input_folder = "frames_spi_usa_spring_ensemble"
output_gif = "spi_animation_usa_spring_hist_ensemble.gif"

# Duration in MILLISECONDS
# 200   = 0.2 seconds (Fast)
# 500   = 0.5 seconds (Normal)
# 1000  = 1.0 second  (Slow)
# 2000  = 2.0 seconds (Very Slow)
frame_duration_ms = 1000 

# ==========================================
# 2. LOAD IMAGES
# ==========================================
# Find all frames
filenames = sorted(glob.glob(os.path.join(input_folder, "frame_*.png")))

if not filenames:
    print("No frames found!")
    exit()

print(f"Found {len(filenames)} frames. Loading...")

# Open images using PIL
# We open the first one specifically to establish the base
frames = [Image.open(f) for f in filenames]

# ==========================================
# 3. SAVE GIF
# ==========================================
print(f"Saving GIF with {frame_duration_ms}ms per frame...")

# This saves the first frame and appends the rest
frames[0].save(
    output_gif,
    format="GIF",
    append_images=frames[1:],
    save_all=True,
    duration=frame_duration_ms,
    loop=0  # 0 means loop forever
)

print(f"Done! Saved to: {output_gif}")

Found 10 frames. Loading...
Saving GIF with 1000ms per frame...
Done! Saved to: spi_animation_usa_spring_hist_ensemble.gif
